# Filtering for cleaner text features

_Feature Engineering_

**Throw away the words that carry no signal — stopwords, the too-common and too-rare, and word variants — so your bag-of-words is small and meaningful.**

A bag-of-words turns each document into a long vector of word counts. But most of those words are useless. A few function words appear so often they drown out everything; a long tail of words appears once and is probably noise. Filtering deletes the useless columns so the useful ones stand out.

       The book gives three complementary filters:

---

This notebook is a practice scaffold for the **Filtering for cleaner text features** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# Lesson: Text filtering — shrink + clean a bag-of-words vocabulary.
# A plain bag-of-words counts EVERY word. Two kinds of words just add junk:
#   1. STOPWORDS  — "the", "a", "is", "and" — appear everywhere, in BOTH
#      classes, so they carry no signal. They sit at the TOP of the counts.
#   2. RARE one-off tokens — a word that shows up in only ONE document
#      (a typo, a name) — can't generalize and just bloats the vocabulary.
# FIX: drop English stopwords (stop_words='english') and drop words that appear
# in fewer than 2 documents (min_df=2). The vocabulary gets much SMALLER, the
# top features become real CONTENT words, and the SAME classifier matches or
# beats the unfiltered one with far fewer features.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# ---- STEP 1: Load real data --------------------------------------------------
# A small REAL inline corpus of short product reviews. No downloads.
# label 1 = positive, label 0 = negative. Note every sentence is padded with
# ordinary English stopwords (the, a, is, and, this, ...) exactly like real text,
# and a few one-off rare words (gizmo, thingy, acme, brandx) appear just once.
texts = [
    "the product is great and i really love it",        # 1
    "a wonderful gadget the build is excellent",        # 1
    "this is a fantastic and reliable device gizmo",    # 1
    "i love the design it is great and sturdy",         # 1
    "the gadget is wonderful and works perfectly acme", # 1
    "an excellent reliable product i really love this",  # 1
    "the product is terrible and i really hate it",     # 0
    "a horrible gadget the build is awful thingy",      # 0
    "this is a broken and unreliable device",           # 0
    "i hate the design it is terrible and flimsy",      # 0
    "the gadget is horrible and breaks constantly",     # 0
    "an awful unreliable product i really hate brandx", # 0
]
y = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0])

# Hold out 4 brand-new sentences the model never sees during training.
test_texts = [
    "the product is great and reliable",   # 1
    "a wonderful excellent gadget",        # 1
    "this gadget is terrible and awful",   # 0
    "the design is horrible and broken",   # 0
]
test_y = np.array([1, 1, 0, 0])


def top_tokens(matrix, vocab, k=10):
    """Return the k most frequent (token, total_count) pairs, biggest first."""
    totals = matrix.sum(axis=0)
    order = np.argsort(totals)[::-1][:k]
    return [vocab[i] for i in order], [int(totals[i]) for i in order]


# ---- STEP 2 + 3: RAW (unfiltered) bag-of-words -> reproduce the PROBLEM -------
# Plain CountVectorizer: keep every word, including stopwords and one-off tokens.
raw_vec = CountVectorizer()
Xr = raw_vec.fit_transform(texts).toarray()
raw_vocab = raw_vec.get_feature_names_out()
raw_words, raw_counts = top_tokens(Xr, raw_vocab)
print("STEP 3 PROBLEM  raw (unfiltered) bag-of-words")
print(f"  vocabulary size = {len(raw_vocab)} words")
print(f"  top tokens (mostly meaningless stopwords): {list(zip(raw_words, raw_counts))}")

clf = LogisticRegression(max_iter=1000, random_state=0)
clf.fit(Xr, y)
raw_acc = (clf.predict(raw_vec.transform(test_texts).toarray()) == test_y).mean()
print(f"  held-out accuracy = {raw_acc:.3f} (using all {len(raw_vocab)} features)")

# ---- STEP 4: Apply the FILTER, then look at the engineered vocabulary ---------
# stop_words='english' uses scikit-learn's BUILT-IN english stopword list (no
# nltk). min_df=2 drops any word seen in fewer than 2 documents (kills one-offs).
filt_vec = CountVectorizer(stop_words="english", min_df=2)
Xf = filt_vec.fit_transform(texts).toarray()
filt_vocab = filt_vec.get_feature_names_out()
filt_words, filt_counts = top_tokens(Xf, filt_vocab)
print("\nSTEP 4 FIX      filtered bag-of-words (stop_words='english', min_df=2)")
print(f"  vocabulary size = {len(filt_vocab)} words")
print(f"  top tokens (now real content words): {list(zip(filt_words, filt_counts))}")

# ---- STEP 5: Show the FIX — SAME classifier on the smaller feature set --------
clf2 = LogisticRegression(max_iter=1000, random_state=0)
clf2.fit(Xf, y)
filt_acc = (clf2.predict(filt_vec.transform(test_texts).toarray()) == test_y).mean()
print(f"  held-out accuracy = {filt_acc:.3f} (using only {len(filt_vocab)} features)")

# ---- PLOTS -------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

# Top tokens BEFORE filtering: stopwords dominate the top of the bar chart.
ax[0].barh(range(len(raw_words))[::-1], raw_counts, color="#c0392b")
ax[0].set_yticks(range(len(raw_words))[::-1]); ax[0].set_yticklabels(raw_words)
ax[0].set_title("BEFORE: top tokens are stopwords\n(the, a, is, and ... no signal)")
ax[0].set_xlabel("total count in corpus")

# Top tokens AFTER filtering: real content words rise to the top.
ax[1].barh(range(len(filt_words))[::-1], filt_counts, color="#27ae60")
ax[1].set_yticks(range(len(filt_words))[::-1]); ax[1].set_yticklabels(filt_words)
ax[1].set_title("AFTER: top tokens are content words\n(gadget, product, love, hate ...)")
ax[1].set_xlabel("total count in corpus")

# Vocabulary size + accuracy, before vs after, side by side.
x = np.arange(2)
ax[2].bar(x - 0.2, [len(raw_vocab), len(filt_vocab)], width=0.4,
          color="#8e44ad", label="vocab size")
ax2b = ax[2].twinx()
ax2b.bar(x + 0.2, [raw_acc, filt_acc], width=0.4, color="#2980b9", label="accuracy")
ax[2].set_xticks(x); ax[2].set_xticklabels(["BEFORE\n(raw)", "AFTER\n(filtered)"])
ax[2].set_ylabel("vocabulary size (purple)"); ax2b.set_ylabel("accuracy (blue)")
ax2b.set_ylim(0, 1.05)
ax[2].set_title("vocab size shrinks, accuracy holds/improves")
plt.tight_layout(); plt.show()

# ---- One-line summary --------------------------------------------------------
print(f"\nPROBLEM (raw): vocab={len(raw_vocab)}, acc={raw_acc:.3f}   ->   "
      f"FIX (filtered): vocab={len(filt_vocab)}, acc={filt_acc:.3f}")

## Reference implementation — scikit-learn, nltk, pandas

In [ ]:
# pip install scikit-learn nltk pandas
# Dataset: Yelp reviews (Yelp Dataset Challenge). Get it via the book's repo:
#   github.com/alicezheng/feature-engineering-book
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# --- load a slice of Yelp reviews into a DataFrame ---
with open('yelp_academic_dataset_review.json') as f:
    reviews = pd.DataFrame([json.loads(line) for line in f][:10000])
texts = reviews['text']

# --- 1) Bag-of-words with NO filtering: huge, noisy vocabulary ---
bow = CountVectorizer()
X_raw = bow.fit_transform(texts)
print("no filtering  : vocab size =", len(bow.get_feature_names_out()))

# --- 2) Remove English stopwords (ultra-common function words) ---
bow_stop = CountVectorizer(stop_words='english')
X_stop = bow_stop.fit_transform(texts)
print("stopwords     : vocab size =", len(bow_stop.get_feature_names_out()))

# --- 3) Frequency-based filtering: drop the rare tail (min_df) AND the
#        too-common tail (max_df). This prunes both ends of the curve. ---
bow_freq = CountVectorizer(stop_words='english', min_df=2, max_df=0.9)
X_freq = bow_freq.fit_transform(texts)
print("stop+min/max  : vocab size =", len(bow_freq.get_feature_names_out()))
# min_df=2  -> drop words appearing in fewer than 2 documents (typos, noise)
# max_df=0.9-> drop words appearing in more than 90% of documents (de-facto stopwords)

# --- 4) Stemming with the Porter stemmer: collapse variants to a root ---
import nltk
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()
print(stemmer.stem('flowers'))   # -> 'flower'
print(stemmer.stem('swimming'))  # -> 'swim'
print(stemmer.stem('news'))      # -> 'new'   (over-stemming! 'news' != 'new')

# Plug stemming into the tokenizer so variants merge into one feature:
import re
def stemmed_tokens(doc):
    tokens = re.findall(r"[a-z]+", doc.lower())
    return [stemmer.stem(t) for t in tokens]

bow_stem = CountVectorizer(tokenizer=stemmed_tokens, stop_words='english',
                           min_df=2, max_df=0.9)
X_stem = bow_stem.fit_transform(texts)
print("stemmed       : vocab size =", len(bow_stem.get_feature_names_out()))

# Fit the vocabulary on TRAIN only, then transform val/test with it (no leakage):
# bow_freq.fit(train_texts); X_val = bow_freq.transform(val_texts)

## Visualize the data & results

_Building a bag-of-words on five short reviews, how much does the vocabulary (the number of feature columns) shrink as we add (1) stopword removal and (2) frequency filtering with min_df=2?_

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "The food was great and the service was great",
    "The pizza is the best pizza in the city",
    "Service was slow but the food was good",
    "A great place with great food and great drinks",
    "The best food and the best service ever",
]

# 1) No filtering: full bag-of-words vocabulary
bow = CountVectorizer()
X0 = bow.fit_transform(corpus)
print("no filtering :", len(bow.get_feature_names_out()))   # 18

# 2) Remove English stopwords (the, a, is, was, and, ...)
bow_stop = CountVectorizer(stop_words='english')
X1 = bow_stop.fit_transform(corpus)
print("stopwords    :", len(bow_stop.get_feature_names_out()))  # 10

# 3) Stopwords + frequency filtering: drop the rare tail (min_df=2)
bow_freq = CountVectorizer(stop_words='english', min_df=2)
X2 = bow_freq.fit_transform(corpus)
print("stop+min_df  :", len(bow_freq.get_feature_names_out()))  # 4
print("survivors    :", sorted(bow_freq.get_feature_names_out()))
# ['best', 'food', 'great', 'service']

# Most-frequent tokens before vs after stopword removal (total counts):
def top_tokens(vec, X, k=4):
    counts = np.asarray(X.sum(axis=0)).ravel()
    feats = vec.get_feature_names_out()
    order = counts.argsort()[::-1][:k]
    return [(feats[i], int(counts[i])) for i in order]
print("top, no filter:", top_tokens(bow, X0))   # the:8, great:5, was:4, food:4
print("top, stopwords:", top_tokens(bow_stop, X1))  # great:5, food:4, service:3, best:3

import matplotlib.pyplot as plt
labels = ["no filtering", "stopwords", "stopwords+min_df=2"]
sizes = [len(bow.get_feature_names_out()),
         len(bow_stop.get_feature_names_out()),
         len(bow_freq.get_feature_names_out())]
plt.bar(labels, sizes, color=["#ffb454", "#4ea1ff", "#7ee787"])
plt.ylabel("vocabulary size V"); plt.title("Filtering shrinks the bag-of-words")
for i, v in enumerate(sizes):
    plt.text(i, v + 0.2, str(v), ha="center")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A corpus has $D = 100$ documents. The word "data" appears in 98 of them and the word "quux" appears in just 1. You set CountVectorizer(min_df=2, max_df=0.9). Which of the two words survives, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute document frequencies: $\mathrm{df}(\text{data}) = 98$, $\mathrm{df}(\text{quux}) = 1$. — _Filtering decisions use $\mathrm{df}$, the number of documents containing the word._
- Apply max_df: $\mathrm{max\_df} = 0.9 \times 100 = 90$. "data" has $\mathrm{df} = 98 \gt 90$, so it is dropped (too common). — _A word in 98% of documents cannot separate documents — it behaves like a stopword._
- Apply min_df: $\mathrm{min\_df} = 2$. "quux" has $\mathrm{df} = 1 \lt 2$, so it is dropped (too rare). — _A word in a single document is likely noise; a model cannot learn from one example._

**Answer:** Neither survives. "data" is cut by max_df (too common, the high tail), and "quux" is cut by min_df (too rare, the low tail). The filter prunes both ends of the word-frequency curve.

</details>

**Problem 2.** You are building a sentiment classifier and you apply stop_words='english'. Reviews like "not good" and "good" now look identical. What went wrong, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the stopword list: standard English lists include negation words like "not", "no", "never". — _Generic stopword lists are built for topic/search tasks, not sentiment._
- Realize that removing "not" turns "not good" into "good", flipping the label. — _Negation words are low-information for topics but high-information for polarity._
- Customize the stopword list: keep negation words (and other polarity-bearing words) for sentiment. — _Stopword lists are domain-specific; the right list depends on the task._

**Answer:** The generic English stopword list removed "not", erasing the negation. Fix: customize the list for sentiment by keeping negation words (e.g. remove them from the stopword set). Stopword filtering is domain-specific.

</details>

**Problem 3.** A Porter stemmer maps "universities" and "university" both to "univers", which helps. But it also maps "news" to "new". Explain the trade-off and one safer alternative.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Stemming chops suffixes by rule, with no dictionary. Merging "universities"/"university" is good: same meaning, now one denser feature. — _Collapsing true variants strengthens the count and reduces vocabulary size._
- But "news" &rarr; "new" merges two unrelated words ("news" the noun, "new" the adjective). — _Rule-based stemming over-collapses because it ignores meaning — the classic Porter pitfall._
- Use lemmatization instead, which uses a dictionary and part-of-speech, so "news" stays "news". — _Lemmatization is meaning-aware and avoids these spurious merges, at higher cost._

**Answer:** Stemming trades precision for denser features: good merges (university variants) come with bad ones (news&rarr;new). A safer, dictionary-aware alternative is lemmatization, which keeps "news" distinct from "new".

</details>